In [ ]:
import os
import sys
# Agrega la ruta raíz del proyecto si no está
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import pandas as pd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import openpyxl
from configs.manager_configuracion import *
from services.Utils.utilidades import *
from services.Carga.cargar_datos_csv import * 
from services.Graficado.base.Gra_mapa_cartopy import graficar_mapa_cartopy
from services.Graficado.base.Gra_mapa_topografia import graficar_mapa_topografico
from services.Graficado.base.Gra_batimetria_en_mapa import graficar_batimetria_en_mapa        
from services.Graficado.base.Gra_trayectorias_de_sonda import graficar_trayectorias_de_sonda
from services.Graficado.base.Gra_dar_formato_a_figuras import *
from services.Utils.convertir_coordenadas import convertir_cualquier_coordenada_a_grados_decimales
from services.Utils.get_coordenadas_del_texto import get_coordenadas_del_texto

In [ ]:
seriales_sondas = ["4858221", "4878218", "4876187", "4878205", "4878196"] # 1 de  diciembre
# seriales_sondas = ["4866704", "4878221", "4878503", "4876191", "4876190"] # 26 de diciembre y 2 de enero
# seriales_sondas = ["4878219", "4878190", "4878213", "4878203", "4876177"] # 19 de febrero 
# seriales_sondas = ["4878214", "4878152", "4876178", "4878225", "4878504"] # 20 de febrero

In [ ]:
# Leer el excel del ASM (allí están las coordenadas de la maniobra)
# Las coordenadas de la maniobra es la coordenada de la primera transmisión @ la hora de la primera transmisión - 2 minutos
# La coordenada del plan es la coordenada de la primera transmisión cortada a 3 decimales @ la hora que se propone

carpeta = "C:/Users/Atmosfera/Desktop/datos_procesados"
archivo = "base_de_datos_general.xlsx"
df = pd.read_excel(os.path.join(carpeta, archivo), sheet_name="campanias")

df.reset_index(inplace=True, drop=True)
df.dropna(subset=["serial_boya"], inplace=True)
df["serial_boya"] = df["serial_boya"].astype(str)

# Filtro del dataframe para quedarnos solo con los seriales de las sondas que nos interesan
df_filtrado = df[df["serial_boya"].isin(seriales_sondas)]
# Convertir la columna de fecha_y_hora_de_maniobra a formato datetime de pandas
df_filtrado["fecha_y_hora_de_maniobra"] = pd.to_datetime(df_filtrado["fecha_y_hora_de_maniobra"])

In [ ]:
# df_filtrado.info()

In [ ]:
# Ordenar el df por la fecha de la maniobra
df_filtrado.sort_values(by="fecha_y_hora_de_maniobra", inplace=True)
# Crear las posiciones lat y lon con la lat_maniobra y lon_maniobra del excel y el pto de embarque
lat_maniobras = [convertir_cualquier_coordenada_a_grados_decimales(coord) for coord in  df_filtrado["lat_maniobra"].tolist()]
lon_maniobras = [convertir_cualquier_coordenada_a_grados_decimales(coord) for coord in  df_filtrado["lon_maniobra"].tolist()]
pto_embarque = df_filtrado["sitio_de_embarque"].iloc[0] # tomo la salida d

# Crear la ruta: pto de salida -> maniobras ordenadas -> pto de llegada
df_embarque = pd.read_excel(os.path.join(carpeta, archivo), sheet_name="sitios_de_embarque")
df_embarque.dropna(subset=["nombre"], inplace=True)
df_embarque_filtrado = df_embarque[df_embarque["nombre"] == pto_embarque]
lat_salida = convertir_cualquier_coordenada_a_grados_decimales(df_embarque_filtrado["latitud"].iloc[0])
lon_salida = convertir_cualquier_coordenada_a_grados_decimales(df_embarque_filtrado["longitud"].iloc[0])

ruta_lon = [lon_salida] + lon_maniobras + [lon_salida]
ruta_lat = [lat_salida] + lat_maniobras + [lat_salida]

In [ ]:
datos_de_topografia = cargar_datos_de_topografia()
datos_de_topografia = recortar_datos_de_topografia_a_area_de_estudio(datos_de_topografia, -98, -90, 18, 26)
datos_de_topografia = reemplazar_valores_de_topografia(datos_de_topografia, valor_a_reemplazar=-10, nuevo_valor=-10)
datos_de_batimetria = cargar_datos_de_batimetria()


In [ ]:
lon_txt_usadas = []
lat_txt_usadas = []

colores_doris = ['blue', 'green', 'purple', 'cyan', 'magenta', 'yellow', 'lime', 'teal', 'pink', 'gray'] # definir una lista de colores para las sondas

lon_min = np.mean(ruta_lon) - 1
lon_max = np.mean(ruta_lon) + 1
lat_min = np.mean(ruta_lat) - 1
lat_max = np.mean(ruta_lat) + 1

fig, ax = plt.subplots(figsize=(16, 7), subplot_kw={'projection': ccrs.PlateCarree()})
graficar_mapa_cartopy(ax, lon_min, lon_max, lat_min, lat_max)
graficar_mapa_topografico(axe=ax, topografia=datos_de_topografia, lon_min=lon_min, lon_max=lon_max, lat_min=lat_min, lat_max=lat_max)
graficar_batimetria_en_mapa(ax, datos_de_batimetria = datos_de_batimetria)

ax.plot(ruta_lon, ruta_lat, color='blue', linewidth=2, transform=ccrs.PlateCarree(), zorder=5) # graficar la ruta de la maniobra
for serial, lon, lat, color in zip(seriales_sondas, lon_maniobras, lat_maniobras, colores_doris):
    ax.scatter(lon, lat, c=color, edgecolors='black', linewidths=1, s=35, label=serial, transform=ccrs.PlateCarree(), zorder=10)
    
ax.scatter(lon_salida, lat_salida, c='orange', edgecolors='black', linewidths=1, s=35, transform=ccrs.PlateCarree(), zorder=10)


lon_txt, lat_txt = get_coordenadas_del_texto(lon_salida, lat_salida, pto_embarque)
plt.text(lon_txt, lat_txt, f"{pto_embarque}", fontsize=12, fontweight='bold', color='black', 
            bbox=dict(facecolor='white', alpha=.3, edgecolor='none', pad=5),transform=ccrs.PlateCarree(), zorder=11)

ax.legend()
plt.show();

In [ ]:
ruta_a_figura = guardar_figura(figura= fig, 
                               ruta_a_carpeta = "C:\\Users\\Atmosfera\\Desktop\\datos_procesados\\doris\\202602",
                               nombre_archivo="1_diciembre", 
                               formato="png", 
                               resolucion=300)
